In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install pandas openpyxl torch transformers tqdm

In [ ]:
"""
T5 Model Fine-tuning for Keywords to Sentence Generation
Input: Excel file with columns 'text' and 'extracted_keywords'
Output: Saved fine-tuned model
"""

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.optim import AdamW
from tqdm import tqdm
import os

# Configuration
class Config:
    EXCEL_FILE = "/content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Keyword to sentence/Text and Keywords - With the sentences - 50k.xlsx"  # Change this to your Excel file path
    MODEL_NAME = "t5-small"  # Options: t5-small, t5-base, t5-large
    OUTPUT_DIR = "/content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Keyword to sentence/fine_tuned_t5_model"
    MAX_SOURCE_LENGTH = 128  # Max length for keywords
    MAX_TARGET_LENGTH = 256  # Max length for generated sentences
    BATCH_SIZE = 8
    EPOCHS = 3
    LEARNING_RATE = 5e-5
    TRAIN_SPLIT = 0.9  # 90% training, 10% validation

# Custom Dataset Class
class KeywordToSentenceDataset(Dataset):
    def __init__(self, data, tokenizer, max_source_len, max_target_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Add prefix for T5 (helps model understand the task)
        source_text = f"generate sentence from keywords: {row['extracted_keywords']}"
        target_text = row['text']

        # Tokenize source (keywords)
        source = self.tokenizer(
            source_text,
            max_length=self.max_source_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Tokenize target (sentence)
        target = self.tokenizer(
            target_text,
            max_length=self.max_target_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': source['input_ids'].squeeze(),
            'attention_mask': source['attention_mask'].squeeze(),
            'labels': target['input_ids'].squeeze()
        }

def load_and_prepare_data(excel_path):
    """Load data from Excel and prepare for training"""
    print(f"Loading data from {excel_path}...")
    df = pd.read_excel(excel_path)

    # Check if required columns exist
    if 'text' not in df.columns or 'extracted_keywords' not in df.columns:
        raise ValueError("Excel file must have 'text' and 'extracted_keywords' columns")

    # Remove rows with missing values
    df = df.dropna(subset=['text', 'extracted_keywords'])

    # Convert to string type
    df['text'] = df['text'].astype(str)
    df['extracted_keywords'] = df['extracted_keywords'].astype(str)

    print(f"Loaded {len(df)} samples")
    print(f"\nSample data:")
    print(df.head(2))

    return df

def train_model(model, train_loader, val_loader, optimizer, device, epochs):
    """Training loop"""
    model.train()
    best_val_loss = float('inf')

    for epoch in range(epochs):
        print(f"\n{'='*50}")
        print(f"Epoch {epoch + 1}/{epochs}")
        print(f"{'='*50}")

        # Training
        total_train_loss = 0
        train_progress = tqdm(train_loader, desc="Training")

        for batch in train_progress:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            total_train_loss += loss.item()

            # Backward pass
            loss.backward()
            optimizer.step()

            train_progress.set_postfix({'loss': loss.item()})

        avg_train_loss = total_train_loss / len(train_loader)
        print(f"Average Training Loss: {avg_train_loss:.4f}")

        # Validation
        model.eval()
        total_val_loss = 0

        with torch.no_grad():
            val_progress = tqdm(val_loader, desc="Validation")
            for batch in val_progress:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                total_val_loss += outputs.loss.item()
                val_progress.set_postfix({'loss': outputs.loss.item()})

        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Average Validation Loss: {avg_val_loss:.4f}")

        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            print(f"New best model! Saving...")

        model.train()

    return model

def main():
    """Main training pipeline"""
    config = Config()

    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load data
    df = load_and_prepare_data(config.EXCEL_FILE)

    # Split into train and validation
    train_size = int(len(df) * config.TRAIN_SPLIT)
    train_df = df[:train_size]
    val_df = df[train_size:]

    print(f"\nTraining samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")

    # Load tokenizer and model
    print(f"\nLoading {config.MODEL_NAME} model and tokenizer...")
    tokenizer = T5Tokenizer.from_pretrained(config.MODEL_NAME)
    model = T5ForConditionalGeneration.from_pretrained(config.MODEL_NAME)
    model.to(device)

    # Create datasets
    train_dataset = KeywordToSentenceDataset(
        train_df, tokenizer, config.MAX_SOURCE_LENGTH, config.MAX_TARGET_LENGTH
    )
    val_dataset = KeywordToSentenceDataset(
        val_df, tokenizer, config.MAX_SOURCE_LENGTH, config.MAX_TARGET_LENGTH
    )

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, batch_size=config.BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=config.BATCH_SIZE, shuffle=False
    )

    # Optimizer
    optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE)

    # Train
    print("\nStarting training...")
    model = train_model(
        model, train_loader, val_loader, optimizer, device, config.EPOCHS
    )

    # Save model and tokenizer
    print(f"\nSaving model to {config.OUTPUT_DIR}...")
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    model.save_pretrained(config.OUTPUT_DIR)
    tokenizer.save_pretrained(config.OUTPUT_DIR)

    print("\n" + "="*50)
    print("Training completed successfully!")
    print(f"Model saved to: {config.OUTPUT_DIR}")
    print("="*50)

if __name__ == "__main__":
    main()

Using device: cuda
Loading data from /content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Keyword to sentence/Text and Keywords - With the sentences - 50k.xlsx...
Loaded 65443 samples

Sample data:
                                                text  \
0  Wall St. Bears Claw Back Into the Black (Reute...   
1  Carlyle Looks Toward Commercial Aerospace (Reu...   

                                  extracted_keywords  
0  wall, green, bears, black, reuters, st, claw, ...  
1  carlyle, firm, reuters, market, aerospace, ind...  

Training samples: 58898
Validation samples: 6545

Loading t5-small model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

...

Average Validation Loss: 0.2022
New best model! Saving...

Epoch 3/3


Training: 100%|██████████| 7363/7363 [31:10<00:00,  3.94it/s, loss=0.24]


Average Training Loss: 0.4192


Validation: 100%|██████████| 819/819 [01:05<00:00, 12.50it/s, loss=0.0919]


Average Validation Loss: 0.1990
New best model! Saving...

Saving model to /content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Keyword to sentence/fine_tuned_t5_model...

Training completed successfully!
Model saved to: /content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Keyword to sentence/fine_tuned_t5_model
